In [42]:
# ==========================================================
# FIFA ETL - PART 1
# Setup + Upload Files + Load Teams
# ==========================================================

import pandas as pd
import os
import re
from google.colab import files

print("="*60)
print("FIFA ETL PIPELINE")
print("="*60)

print("\nUpload these files:")
print("1. teams.csv")
print("2. player stats attacking.txt")
print("3. player stats distribution.txt")
print("4. player stats defending.txt")
print("5. player stats discipline.txt")
print("6. player stats goalkeeping.txt")
print("7. player stats goalsbook.txt")

uploaded = files.upload()

print("\nUploaded Files:")
for f in uploaded.keys():
    print("✓", f)

os.makedirs("output", exist_ok=True)


FIFA ETL PIPELINE

Upload these files:
1. teams.csv
2. player stats attacking.txt
3. player stats distribution.txt
4. player stats defending.txt
5. player stats discipline.txt
6. player stats goalkeeping.txt
7. player stats goalsbook.txt


Saving player stats attacking.txt to player stats attacking.txt
Saving player stats defending.txt to player stats defending.txt
Saving player stats discipline.txt to player stats discipline.txt
Saving player stats distribution.txt to player stats distribution.txt
Saving player stats goalkeeping.txt to player stats goalkeeping.txt
Saving player stats goalsbook.txt to player stats goalsbook.txt
Saving teams.csv to teams.csv

Uploaded Files:
✓ player stats attacking.txt
✓ player stats defending.txt
✓ player stats discipline.txt
✓ player stats distribution.txt
✓ player stats goalkeeping.txt
✓ player stats goalsbook.txt
✓ teams.csv


In [43]:
# ==========================================================
# LOAD TEAMS
# ==========================================================

# Find teams.csv automatically
team_file = None

for f in uploaded.keys():
    if "team" in f.lower() and f.lower().endswith(".csv"):
        team_file = f
        break

if team_file is None:
    raise Exception("teams.csv not found!")

teams = pd.read_csv(team_file)

print("\nLoaded:", team_file)

print("\nColumns detected:")
print(list(teams.columns))


Loaded: teams.csv

Columns detected:
['team_id', 'team_name', 'team_code']


In [44]:
# ==========================================================
# CLEAN TEAM HEADERS
# ==========================================================

teams.columns = (
    teams.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(teams.head())

   team_id  team_name team_code
0        1    Algeria       ALG
1        2  Argentina       ARG
2        3  Australia       AUS
3        4    Austria       AUT
4        5    Belgium       BEL


In [45]:
# ==========================================================
# FIFA CODE -> TEAM_ID LOOKUP
# ==========================================================

required = ["team_id", "team_code"]

for col in required:
    if col not in teams.columns:
        raise Exception(f"Column '{col}' missing in teams.csv")

team_lookup = {}

for _, row in teams.iterrows():
    code = str(row["team_code"]).strip().upper()
    team_lookup[code] = int(row["team_id"])

print("\nTeam Lookup Created!")

print("Example:")

for k in list(team_lookup.keys())[:10]:
    print(k, "->", team_lookup[k])

print("\nTotal Teams:", len(team_lookup))


Team Lookup Created!
Example:
ALG -> 1
ARG -> 2
AUS -> 3
AUT -> 4
BEL -> 5
BIH -> 6
BRA -> 7
CPV -> 8
CAN -> 9
COL -> 10

Total Teams: 48


In [46]:
# ==========================================================
# DETECT PLAYER FILES
# ==========================================================

text_files = {}

for f in uploaded.keys():

    name = f.lower()

    if not name.endswith(".txt"):
        continue

    if "attacking" in name:
        text_files["attacking"] = f

    elif "distribution" in name:
        text_files["distribution"] = f

    elif "defending" in name:
        text_files["defending"] = f

    elif "discipline" in name:
        text_files["discipline"] = f

    elif "goalkeeping" in name:
        text_files["goalkeeping"] = f

    if "goalsbook" in name:
        text_files["goalsbook"] = f

print("\nDetected Files")

for k,v in text_files.items():
    print(k, "->", v)

if len(text_files) != 6:
    print("\n⚠ WARNING")
    print("Expected 5 player txt files.")


Detected Files
attacking -> player stats attacking.txt
defending -> player stats defending.txt
discipline -> player stats discipline.txt
distribution -> player stats distribution.txt
goalkeeping -> player stats goalkeeping.txt
goalsbook -> player stats goalsbook.txt


In [47]:
# ==========================================================
# PART 2
# Generic FIFA TXT Parser
# ==========================================================

import re

# ----------------------------------------------------------
# Number of statistics in each file
# ----------------------------------------------------------

FILE_CONFIG = {

    "attacking": {
    "stats": 10,
    "columns": [
        "assists",
        "attempts_on_target",
        "attempts_at_goal",
        "attempts_at_goal_conv_rate",
        "attempts_inside_penalty",
        "attempts_outside_penalty",
        "headed_attempts_at_goal",
        "xg",
        "xg_efficiency",
        "corners"
    ]
},

    "distribution": {
        "stats": 10,
        "columns": [
            "passes",
            "passes completed",
            "passing_accuracy",
            "crosses",
            "crossing_accuracy",
            "take ons completed",
            "defensive_linebreaks_attempted",
            "defensive_linebreaks_accuracy",
            "switches_of_play_attempted",
            "switches_of_play_accuracy"
        ]
    },

    "defending": {
        "stats": 4,
        "columns": [
            "own_goals",
            "forced_turnovers",
            "defensive_pressures_applied",
            "defensive_pressures_directly_applied"
        ]
    },

    "discipline": {
        "stats": 6,
        "columns": [
            "fouls_against",
            "fouls_for",
            "yellow_cards",
            "red_cards",
            "indirect_red_cards",
            "offsides"
        ]
    },

    "goalkeeping": {
        "stats": 3,
        "columns": [
            "goalkeeper_saves",
            "goalkeeper_action_inside_penalty_area",
            "goalkeeper_action_outside_penalty_area"
        ]
    },

    "goalsbook": {
        "stats": 3,
        "columns": [
            "goals",
            "assists",
            "minutes_played"
        ]
    }

}


# ----------------------------------------------------------
# Helper
# ----------------------------------------------------------

def clean_lines(lines):

    cleaned = []

    for line in lines:

        line = line.strip()

        if line == "":
            continue

        cleaned.append(line)

    return cleaned


# ----------------------------------------------------------
# Main Parser
# ----------------------------------------------------------

def parse_file(filepath, category):

    config = FILE_CONFIG[category]

    stat_count = config["stats"]

    stat_columns = config["columns"]

    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.readlines()

    lines = clean_lines(raw)

    records = []

    i = 0

    while i < len(lines):

        # --------------------------
        # Rank
        # --------------------------

        if not re.fullmatch(r"\d+", lines[i]):
            i += 1
            continue

        rank = int(lines[i])
        i += 1

        # --------------------------
        # Player
        # --------------------------

        if i >= len(lines):
            break

        player = lines[i]
        i += 1

        # --------------------------
        # Team
        # --------------------------

        team_name = lines[i]
        i += 1

        fifa_code = lines[i].upper()
        i += 1

        # --------------------------
        # Position
        # --------------------------

        position = lines[i]
        i += 1

        # --------------------------
        # Statistics
        # --------------------------

        stats = []

        while len(stats) < stat_count and i < len(lines):

            stats.append(lines[i])

            i += 1

        # --------------------------
        # Build Record
        # --------------------------

        record = {

            "rank": rank,

            "player_name": player,

            "team_name": team_name,

            "fifa_code": fifa_code,

            "position": position

        }

        for c, v in zip(stat_columns, stats):

            record[c] = v

        records.append(record)

    df = pd.DataFrame(records)

    # --------------------------
    # Map Team ID
    # --------------------------

    df["team_id"] = df["fifa_code"].map(team_lookup)

    return df


# ----------------------------------------------------------
# Parse every file
# ----------------------------------------------------------

parsed_tables = {}

for category, filename in text_files.items():

    print("-" * 50)

    print("Parsing:", filename)

    df = parse_file(filename, category)

    parsed_tables[category] = df

    print("Rows Parsed:", len(df))

    print(df.head(3))

print("\n✅ All files parsed successfully!")

--------------------------------------------------
Parsing: player stats attacking.txt
Rows Parsed: 700
   rank      player_name team_name fifa_code position assists  \
0     1    Michael Olise       FRA       FRA       FW       5   
1     2  Martin Odegaard       NOR       NOR       MF       4   
2     2      Brahim Diaz       MAR       MAR       FW       4   

  attempts_on_target attempts_at_goal attempts_at_goal_conv_rate  \
0                  5               17                          0   
1                  3                9                          0   
2                  1                6                          0   

  attempts_inside_penalty attempts_outside_penalty headed_attempts_at_goal  \
0                       8                        9                       0   
1                       5                        4                       0   
2                       4                        2                       0   

     xg xg_efficiency corners  team_id  
0  1.84 

In [48]:
# ==========================================================
# PART 3
# Create players.csv and normalize all tables
# ==========================================================

import pandas as pd

# ----------------------------------------------------------
# STEP 1
# Collect every player from every parsed table
# ----------------------------------------------------------

all_players = []

for category, df in parsed_tables.items():

    temp = df[[
        "player_name",
        "team_id",
        "position"
    ]].copy()

    all_players.append(temp)

players = pd.concat(all_players, ignore_index=True)

# ----------------------------------------------------------
# STEP 2
# Clean Names
# ----------------------------------------------------------

players["player_name"] = (
    players["player_name"]
    .astype(str)
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

players["position"] = players["position"].str.strip()

# ----------------------------------------------------------
# STEP 3
# Remove duplicates
# ----------------------------------------------------------

players = players.drop_duplicates(
    subset=[
        "player_name",
        "team_id"
    ]
)

players = players.sort_values(
    by=[
        "player_name",
        "team_id"
    ]
).reset_index(drop=True)

# ----------------------------------------------------------
# STEP 4
# Generate Player IDs
# ----------------------------------------------------------

players.insert(
    0,
    "player_id",
    range(1, len(players) + 1)
)

print("="*60)
print("PLAYERS TABLE CREATED")
print("="*60)

print(players.head())

print()

print("Total Unique Players :", len(players))

# ----------------------------------------------------------
# STEP 5
# Build lookup
# ----------------------------------------------------------

player_lookup = {}

for _, row in players.iterrows():

    key = (
        row["player_name"],
        row["team_id"]
    )

    player_lookup[key] = row["player_id"]

print("Player Lookup Built.")

PLAYERS TABLE CREATED
   player_id          player_name  team_id position
0          1         Aaron Hickey       38       DF
1          2       Aaron Tshibola       11       MF
2          3    Aaron Wan Bissaka       11       DF
3          4  Abbosbek Fayzullaev       48       MF
4          5   Abdallah Alfakhori       26       GK

Total Unique Players : 1242
Player Lookup Built.


In [49]:
# ==========================================================
# Attach player_id to every parsed table
# ==========================================================

normalized_tables = {}

for category, df in parsed_tables.items():

    temp = df.copy()

    temp["player_id"] = temp.apply(
        lambda r: player_lookup[
            (
                r["player_name"],
                r["team_id"]
            )
        ],
        axis=1
    )

    # remove unwanted columns

    temp = temp.drop(
        columns=[
            "rank",
            "team_name",
            "fifa_code"
        ]
    )

    # move IDs to front

    cols = list(temp.columns)

    cols.remove("player_id")
    cols.remove("team_id")

    temp = temp[
        ["player_id", "team_id"] + cols
    ]

    normalized_tables[category] = temp

print()

print("Normalized Tables Ready")

for k, v in normalized_tables.items():

    print(k, ":", len(v))


Normalized Tables Ready
attacking : 700
defending : 1242
discipline : 1242
distribution : 1242
goalkeeping : 145
goalsbook : 1231


In [50]:
# ==========================================================
# PART 4
# Export CSVs + Validation + ZIP
# ==========================================================

import os
import shutil

OUTPUT_FOLDER = "output"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ----------------------------------------------------------
# Save players.csv
# ----------------------------------------------------------

players.to_csv(
    os.path.join(OUTPUT_FOLDER, "players.csv"),
    index=False
)

# ----------------------------------------------------------
# Save normalized tables
# ----------------------------------------------------------

file_names = {
    "attacking": "player_attacking.csv",
    "distribution": "player_distribution.csv",
    "defending": "player_defending.csv",
    "discipline": "player_discipline.csv",
    "goalkeeping": "player_goalkeeping.csv",
    "goalsbook":"player_goalsbook.csv"
}

print("=" * 60)
print("EXPORTING CSV FILES")
print("=" * 60)

for category, filename in file_names.items():

    df = normalized_tables[category]

    df.to_csv(
        os.path.join(OUTPUT_FOLDER, filename),
        index=False
    )

    print(f"✓ {filename} ({len(df)} rows)")

print("✓ players.csv ({})".format(len(players)))

# ----------------------------------------------------------
# Validation
# ----------------------------------------------------------

print("\n" + "=" * 60)
print("VALIDATION")
print("=" * 60)

all_ok = True

player_ids = set(players["player_id"])

for category, df in normalized_tables.items():

    missing = set(df["player_id"]) - player_ids

    if len(missing) > 0:
        print(f"❌ {category}: Missing Player IDs")
        all_ok = False
    else:
        print(f"✅ {category}: Foreign Keys OK")

if players["player_id"].duplicated().any():
    print("❌ Duplicate Player IDs")
    all_ok = False
else:
    print("✅ Player IDs Unique")

if players[["player_name", "team_id"]].duplicated().any():
    print("❌ Duplicate Players")
    all_ok = False
else:
    print("✅ No Duplicate Players")

# ----------------------------------------------------------
# ZIP everything
# ----------------------------------------------------------

zip_name = "fifa_normalized_dataset"

shutil.make_archive(
    zip_name,
    'zip',
    OUTPUT_FOLDER
)

print("\n" + "=" * 60)

if all_ok:
    print("🎉 DATASET CREATED SUCCESSFULLY!")
else:
    print("⚠ Dataset created with validation issues.")

print("=" * 60)

print("\nGenerated Files:")

for f in sorted(os.listdir(OUTPUT_FOLDER)):
    print("📄", f)

print("\nZIP File:")
print(zip_name + ".zip")

EXPORTING CSV FILES
✓ player_attacking.csv (700 rows)
✓ player_distribution.csv (1242 rows)
✓ player_defending.csv (1242 rows)
✓ player_discipline.csv (1242 rows)
✓ player_goalkeeping.csv (145 rows)
✓ player_goalsbook.csv (1231 rows)
✓ players.csv (1242)

VALIDATION
✅ attacking: Foreign Keys OK
✅ defending: Foreign Keys OK
✅ discipline: Foreign Keys OK
✅ distribution: Foreign Keys OK
✅ goalkeeping: Foreign Keys OK
✅ goalsbook: Foreign Keys OK
✅ Player IDs Unique
✅ No Duplicate Players

🎉 DATASET CREATED SUCCESSFULLY!

Generated Files:
📄 player_attacking.csv
📄 player_defending.csv
📄 player_discipline.csv
📄 player_distribution.csv
📄 player_goalkeeping.csv
📄 player_goalsbook.csv
📄 players.csv

ZIP File:
fifa_normalized_dataset.zip


In [51]:
from google.colab import files

files.download("fifa_normalized_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>